In [7]:
from src.runtime_constants import rr
from src.utils.gembase_utils import GembaseUtils
import json
from src.db import DbConnection

conn = DbConnection()
#conn = DbConnection(rr.ENV_PROD, True)

page = 0
limit = 500

conn.query("""
CREATE TEMPORARY TABLE app_temp_data.erik_update_app_pricing (
app_id_int int unsigned not null,
price decimal(8,2) not null,
initial_price decimal(8,2) not null,
constraint erik_update_app_pricing_pk
primary key (app_id_int)
)
""")

while True:

    print(f"Page: {page}")

    rows_apps = conn.select_all(f"""
    SELECT a.app_id_int, asd.data, a.store
      FROM scraped_data.apps a,
           scraped_data.apps_source_data asd
     WHERE asd.app_id_int = a.app_id_int
       AND (asd.data_type = 'steamspy_data' and a.store = 2
        OR asd.data_type = 'scrap_data' and a.store = 1)
     ORDER BY a.app_id_int
     LIMIT {page * limit}, {limit}
    """)

    if len(rows_apps) == 0:
        break

    page += 1

    bulk_data = []

    for row in rows_apps:

        data = GembaseUtils.decompress(row["data"])
        data_parsed = json.loads(data)

        price = 0
        initial_price = 0

        if row["store"] == 1:
            price = data_parsed["price"]
            initial_price = data_parsed["originalPrice"]
            # "price":0,
            # "free":true,
            # "originalPrice":null,
        elif row["store"] == 2:
            price = data_parsed["price"]
            if price is not None:
                price = int(price) / 100
            initial_price = data_parsed["initialprice"]
            if initial_price is not None:
                initial_price = int(initial_price) / 100
            # "initialprice":"1999",
            # "discount":"75",

        if price is None:
            price = 0
        if initial_price is None:
            initial_price = price

        bulk_data.append((price, initial_price, row["app_id_int"]))
        #break

    conn.bulk("""
    INSERT INTO app_temp_data.erik_update_app_pricing (price, initial_price, app_id_int)
    VALUES (%s, %s, %s)
    """, bulk_data)

    conn.commit()

conn.query("""
UPDATE scraped_data.apps a
INNER JOIN app_temp_data.erik_update_app_pricing t
ON t.app_id_int = a.app_id_int
   SET a.price = t.price,
       a.initial_price = t.initial_price
""")

conn.commit()

conn.close()

Page: 0
Page: 1


In [ ]:
conn.close()